# Session 11: Pause, resume, remember forever

**Week 6 · Session 1 · LangGraph II of III · Project version v8**

Session 10 opened the black box: v7's inbox is a graph you can draw. It still runs blind from start to
finish, forgets everything when the process ends, and has never met a customer twice. Tonight it learns
to stop and wait for a person, to wake up where it slept, and to remember who it is talking to.

<img src="figures/session-map.png" width="900"
     alt="The whole of session 11 on one page. A top strip contrasts v7, which runs blind to the end and
     forgets on exit, with v8, which pauses for a manager, resumes after a restart and remembers the
     customer. Below, three zones. Zone A, durable execution and humans: a checkpointer, inspecting a
     run, interrupt and its four rules, four human-in-the-loop shapes, durable execution, time travel.
     Zone B, memory that outlives the ticket: summarizing with RemoveMessage, the Store, identity as
     runtime context, semantic search. Zone C, conversations: query rewriting for follow-ups and the
     typed event vocabulary. A bottom band says a paused run is just a saved one.">

There are two main topics, and a third part that puts them to work:

| | Topic | Question it answers |
|---|---|---|
| **A** | **Durable execution and humans** | How does a run stop for a person, survive a crash, and get debugged after the fact? |
| **B** | **Memory that outlives the ticket** | What should the assistant remember about a customer, where, and who may read it? |
| **C** | Conversations meet retrieval, and the graph learns to speak | Why follow-ups break RAG, and what the outside world is told while a turn runs |

By the end you will have **v8**: refunds pause for a manager, threads survive restarts, customers are
remembered across tickets, follow-up questions retrieve correctly, and the graph speaks a typed event
vocabulary. `chat.py` moves onto the graph tonight.

## 1. Setup

**One new package, `langgraph-checkpoint-sqlite`.** Both of tonight's stores come from it: `SqliteSaver`
(checkpoints) and `SqliteStore` (customer memory), plus the `sqlite-vec` extension that lets the Store
search memories by meaning. It is in `pyproject.toml`, so after pulling, run `uv sync` once.

The v8 project changes more files than any session so far. The notebook imports the real modules:

| Module | What it does |
|---|---|
| `workflows/inbox/` | the graph gains `intake`, `summarize`, `rewrite`, `approval` and `refund`, a conversation in its state, and `open_inbox()`: checkpoints and memory on disk |
| `workflows/memory.py` | **new**: customer memory. The Store, plus the `recall` and `remember` nodes |
| `workflows/events.py` | **new**: seven typed events and `stream_ticket()`, the generator every caller uses |
| `kb/query.py` | **new**: rewrites a follow-up into a standalone question |
| `prompts/<feature>.yml`, `models.py` | **new**: the session's structural move (section 16) |
| `chat.py` | rewritten as a client of `stream_ticket()`, with manager commands and `--thread` |
| `tools.py`, `shopwise_data.py` | `issue_refund` is idempotent and writes a refund ledger |
| `evals/conversational.yml` | **new**: two-turn conversations, scored by `run_eval.py --followups` |

**Providers, as in session 10.** Answers come from `openrouter:openai/gpt-oss-20b`, and every search and
every memory embeds with Gemini's `gemini-embedding-2`. **The notebook makes about 170 chat-model calls**
and about 60 embedding calls. The committed outputs show what you would have seen.

**The notebook keeps its own save files.** The first cell points the three v8 stores at files named
`s11-notebook-*` in `labs/.state/` and deletes them, so every run starts from the same clean state and
your own `chat.py` tickets are never touched. The variables are read when `config.py` is imported,
which is why they are set before it.

**Docs:** [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) ·
[Interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts) ·
[Memory](https://docs.langchain.com/oss/python/langgraph/add-memory)

In [1]:
import json
import os
import subprocess
import sys
import warnings
from pathlib import Path

from langchain_core._api import LangChainBetaWarning

warnings.filterwarnings("ignore", category=LangChainBetaWarning)
warnings.filterwarnings("ignore", message="IProgress not found")    # fastembed's progress bar

HERE = Path.cwd()
STATE = HERE.parent / ".state"                   # labs/.state, gitignored
NOTEBOOK_FILES = {"SHOPWISE_CHECKPOINT_DB": STATE / "s11-notebook-checkpoints.sqlite",
                  "SHOPWISE_MEMORY_DB": STATE / "s11-notebook-memory.sqlite",
                  "SHOPWISE_REFUND_LEDGER": STATE / "s11-notebook-refunds.jsonl"}
for name, path in NOTEBOOK_FILES.items():
    for leftover in (path, Path(f"{path}-wal"), Path(f"{path}-shm")):
        leftover.unlink(missing_ok=True)         # a clean start, every run (SQLite's -wal/-shm too)
    os.environ[name] = str(path)                 # before config.py is imported

sys.path.insert(0, str(HERE / "project"))        # the v8 snapshot beside this notebook

from config import settings                      # noqa: E402
from kb import store                             # noqa: E402

points = store.shared_client().count(settings.collection).count
store.close_shared()                             # embedded Qdrant: section 2 runs v7 in another process
print(f"{settings.APP_NAME} v{settings.VERSION} · {settings.model_id}")
print(f"knowledge base: {settings.collection} · {points} chunks")
print(f"checkpoints: {settings.CHECKPOINT_DB.name} · memory: {settings.MEMORY_DB.name} · "
      f"refunds: {settings.REFUND_LEDGER.name}")
print(f"a chat message counts as received on {settings.today} (AS_OF)")
assert points, "Empty knowledge base: run `uv run python kb/ingest.py` from project/ first."

ShopWise Support Assistant v8.0.0 · openrouter:openai/gpt-oss-20b
knowledge base: policies__gemini-embedding-2-768 · 70 chunks
checkpoints: s11-notebook-checkpoints.sqlite · memory: s11-notebook-memory.sqlite · refunds: s11-notebook-refunds.jsonl
a chat message counts as received on 2026-08-09 (AS_OF)


Three helpers. `show` prints one definition straight from a project file, so you read the code that
actually runs. `v7` runs session 10's snapshot in its own process, because two versions of the same
modules cannot share one kernel. `TICKETS` holds every ticket both inboxes know.

In [2]:
from inbox import load_all_tickets              # noqa: E402

TICKETS = load_all_tickets()


def show(name, file):
    """Print one top-level definition from a project file: the real code, not a copy."""
    lines = (HERE / "project" / file).read_text().splitlines()
    start = next(i for i, l in enumerate(lines)
                 if l.startswith((f"def {name}(", f"{name} = ", f"class {name}(", f"class {name}:")))
    end = start + 1
    while end < len(lines) and (not lines[end] or lines[end][0] in " )]}"):
        end += 1
    while start > 0 and lines[start - 1].startswith("@"):   # keep decorators such as @contextmanager
        start -= 1
    print("\n".join(lines[start:end]).rstrip())


def v7(*args):
    """Run session 10's inbox.py (v7) in its own process and print what it printed."""
    store.close_shared()
    out = subprocess.run([sys.executable, "inbox.py", *args], cwd=HERE.parent / "session-10" / "project",
                         capture_output=True, text=True)
    print(out.stdout.strip() or out.stderr[-1500:])

## 2. The failures: four missing organs

Session 10 ended with v7 answering a multi-issue ticket in one reply. Run it on four ordinary support
situations and four things are missing.

**Failure 1: no pause.** TCK-M05 is Ayesha asking for the $129 refund on her crackling headphones,
which the shop logged as REF-9001 and never paid. v7's billing desk can see exactly what is owed. It
has no `issue_refund`, because nothing in the graph can stop and wait for the person who must approve
it, so the best it can do is a promise.

In [46]:
v7("TCK-M05")

TCK-M05 from ayesha@example.com: headphones refund


Queue: complaint · urgency medium · a person must handle the money

Hi Ayesha, we have logged your refund request for order ORD-5001 (REF-9001) and it is currently in the requested status. A manager will review and approve or deny the refund shortly.


**Failure 2: no durable ticket memory.** v7 has no conversation at all: one ticket in, one reply out,
and its graph was compiled without a checkpointer. The v7 `chat.py`, the only program that held a
conversation, kept it in a dictionary in the process's memory, and said so on the way out:

In [47]:
v7_chat = (HERE.parent / "session-10" / "project" / "chat.py").read_text()
print(next(line.strip() for line in v7_chat.splitlines() if "lived in this process" in line))
print(next(line.strip() for line in v7_chat.splitlines() if "build_agent(InMemorySaver()" in line))

print(f"{len(usage)} ticket(s) lived in this process's memory, and they're gone now, "
agent = build_agent(InMemorySaver(), stream_tool_steps=False)   # see the docstring


**Failure 3: no customer memory.** A week ago, in TCK-M07, Tanvir wrote that he would much rather have
a replacement than his money back. Today, in TCK-M08, his watch looks lost, and the shipping policy
offers *a replacement or a refund, at your choice*. v7 has never heard of him:

In [5]:
v7("TCK-M08")

TCK-M08 from tanvir@example.com: watch still not here


**Failure 4: the follow-up that retrieves garbage.** Suppose turns did persist. Tanvir asks *"What's the
return window for headphones?"* and gets a cited answer. Then he asks *"And for the watch?"*. Retrieval
embeds those four words as written. Here are the top passages for each, next to the question he
actually meant:

In [48]:
from kb import rag                               # noqa: E402

for question in ("What's the return window for headphones?",
                 "And for the watch?",
                 "What is the return window for a smartwatch?"):
    print(question)
    for doc in rag.search(question, k=3):
        print("   ", rag.label(doc))

What's the return window for headphones?
    [returns-policy · 1. The return window]
    [returns-policy · 4. Starting a return]
    [warranty-policy · 2. How long each product type is covered]
And for the watch?
    [cfr-16-part-700 · p.6]
    [cfr-16-part-700 · p.6]
    [cfr-16-part-700 · p.2]
What is the return window for a smartwatch?
    [returns-policy · 1. The return window]
    [warranty-policy · 2. How long each product type is covered]
    [returns-policy · 4. Starting a return]


The follow-up retrieves passages about watches in general, not about returning one. This is the most
common RAG bug in production, and session 9's scorecard never caught it: every gold question was asked
cold, with no conversation behind it.

*The first two failures turn out to be the same organ, a checkpointer. The fourth only exists because
we fix the second.*

## 3. Where we are on the map

| | v7 (session 10) | v8 (tonight) |
|---|---|---|
| A refund | "a manager will review it" | the run pauses; a manager approves, edits or rejects; the refund is paid once |
| A restart | everything forgotten | the thread resumes exactly where it slept, a paused refund included |
| A bad answer | re-run and hope | replay or fork the run from any saved step |
| A returning customer | a stranger | remembered, and searchable by meaning |
| Who the customer is | from the login | still from the login, now also the key to their memories |
| "And for the watch?" | searched as written | rewritten into a standalone question first |
| While it works | silence, then a reply | a typed event stream: token · progress · tool · citation · interrupt · done · error |

---
# Part A: Durable execution and humans

## 4. A checkpointer is session 6's argument, inside a graph

In session 6 you passed `checkpointer=InMemorySaver()` to `create_agent`, and the agent remembered the
conversation. Session 10 showed that `create_agent` builds a LangGraph graph. So the checkpointer was
always a graph feature, and `StateGraph.compile()` takes the very same parameter:

```python
graph = builder.compile(checkpointer=InMemorySaver())      # session 6's line, one level down
```

Three words you need from here on:

| Word | What it is | ShopWise |
|---|---|---|
| **thread** | one saved conversation, named by `config["configurable"]["thread_id"]` | one ticket |
| **checkpoint** | a snapshot of the whole state, written after every super-step | one row in the save file |
| **super-step** | one tick of the graph: every node that can run now, runs (in parallel), then the state is saved | `split` is one; the four desks together are one |

A checkpointer is chosen by where the snapshots should live:

| Checkpointer | Lives in | Survives a restart? | Used for |
|---|---|---|---|
| `InMemorySaver` | this process's memory | no | notebooks, tests |
| `SqliteSaver` | one file on disk | yes | **ShopWise from tonight** (`labs/.state/checkpoints.sqlite`) |
| `PostgresSaver` | a database server | yes, and shared by many servers | session 13 onwards |

Switching is one line. `open_inbox()` in `workflows/inbox/graph.py` does it for the programs:

In [49]:
show("open_saver", "workflows/inbox/graph.py")
print()
show("open_inbox", "workflows/inbox/graph.py")

@contextmanager
def open_saver(path: Path | None = None):
    """The checkpointer, on disk. Every ticket is a thread in this one file.

    check_same_thread=False because Send runs desks in worker threads, and each
    writes its result through this connection; SqliteSaver locks around it."""
    path = path or settings.CHECKPOINT_DB
    path.parent.mkdir(parents=True, exist_ok=True)
    conn = sqlite3.connect(str(path), check_same_thread=False)
    try:
        yield SqliteSaver(conn, serde=checkpoint_serde())
    finally:
        conn.close()

@contextmanager
def open_inbox():
    """The graph as the programs use it: checkpoints and memory on disk."""
    with open_saver() as saver, open_memory() as store:
        yield build_inbox(checkpointer=saver, store=store)


`checkpoint_serde()` is there because the state now holds ShopWise's own Pydantic objects
(`TriageResult`, `SubIssue`), and a checkpoint stores them as "module, class, fields". Reading one back
means importing that class, which the serializer only does for classes on an allowlist. Leave it out
and every load warns, and a future LangGraph version refuses.

Run TCK-M01, Ayesha's three-issue ticket from session 10, on the graph saved to disk. `stream_ticket`
is section 18's subject; for now it is "run one turn and hand back what happened", and `render` is the
function `chat.py` prints it with.

In [3]:
from chat import render                          # noqa: E402
from middleware import context_for               # noqa: E402
from workflows.events import paused_on, run_config, stream_ticket   # noqa: E402
from workflows.inbox import open_inbox           # noqa: E402

_inbox = open_inbox()
graph = _inbox.__enter__()                        # held open for the whole notebook


def run_ticket(ticket_id, thread=None):
    """One inbox ticket through stream_ticket, printed the way chat.py prints a turn."""
    ticket = TICKETS[ticket_id]
    render(stream_ticket(graph, thread or ticket_id, ticket, context_for(ticket["from"])))


print(TICKETS["TCK-M01"]["body"], "\n")
run_ticket("TCK-M01")

Hi, a few things. My two BrewMaster kettles (ORD-5002) still say processing after more than a week, when will they actually ship? Separately, I asked for a refund on the crackling headphones a week ago (REF-9001) and have heard nothing, has it gone through? And if one of the kettles breaks later on, how long is the warranty on it? 

assistant > · Checking refund eligibility for ORD-5001… · Searching the policies… · checking get_order… · checking search_policy… · Found 4 policy passages · Asking a manager about the refund on ORD-5001… 
⏸  This refund is waiting for a ShopWise manager.
   [manager] ORD-5001: refund $129.00 (paid $129.00) · inside the 30-day return window (21 days since delivery)
   [manager] recommendation: approve
   [manager] /approve · /edit <amount> <reason> · /reject <reason>



The dim lines are progress and tool events from inside the desks. There is **no reply yet**: billing
wants to pay $129, so the `approval` node called `interrupt()` and the run stopped there. Paused is not
finished. Compose has not run, so nothing has been written to the customer.

Now the restart. A **new Python process**, which knows nothing about this kernel, opens the same file
and reads the thread back. It finds the ticket exactly where it stopped: next in line is `approval`,
still holding the manager's question. No model is called:

In [55]:
probe = """
from workflows.events import run_config
from workflows.inbox import open_inbox
with open_inbox() as g:
    snap = g.get_state(run_config("TCK-M01"))
    print(f"{len(snap.values['messages'])} message saved on TCK-M01 (the customer's; no reply yet)")
    print("reply so far:", repr(snap.values.get("reply", "")))
    print("next node:", snap.next)
    for task in snap.tasks:
        for pending in task.interrupts:
            q = pending.value
            print(f"waiting on: refund ${q['amount']:.2f} for {q['order_id']}, recommendation {q['recommendation']}")
"""
out = subprocess.run([sys.executable, "-c", probe], cwd=HERE / "project", capture_output=True, text=True)
print(out.stdout or out.stderr[-1500:])

1 message saved on TCK-M01 (the customer's; no reply yet)
reply so far: ''
next node: ('approval',)
waiting on: refund $129.00 for ORD-5001, recommendation approve



That process inherited this notebook's `SHOPWISE_CHECKPOINT_DB`, so it opened the notebook's file.
`chat.py` and `inbox.py` share `labs/.state/checkpoints.sqlite` the same way, which is why a refund
paused in the inbox can be answered from the chat (section 19).

> **Gotcha: the thread id is the key, and nothing else is.** Same `thread_id`: the conversation
> continues. A new one: a stranger. Forget to pass a config at all and a graph with a checkpointer
> raises `ValueError: Checkpointer requires one or more of the following 'configurable' keys:
> thread_id...`. ShopWise uses the ticket id as the thread id, so the two can never disagree.

## 5. Inspect a run: `get_state` and the history

A saved thread can be read without running anything. `graph.get_state(config)` returns a
`StateSnapshot`. Two of its fields are tonight's main debugging tools:

| Field | Meaning |
|---|---|
| `.values` | the state as it stands: every key the nodes have written |
| `.next` | which nodes run next. **Empty means finished; not empty means paused or stopped halfway** |
| `.tasks` | the pending tasks, each with any `interrupts` waiting on a person, or the `error` it died with |
| `.config` | the config that points at exactly this checkpoint (it carries a `checkpoint_id`) |

In [6]:
snap = graph.get_state(run_config("TCK-M01"))
print("next:", snap.next or "() — finished")
print("keys:", sorted(snap.values))
print("triage:", snap.values["triage"].category, "·", snap.values["triage"].urgency)
for f in sorted(snap.values["findings"], key=lambda f: f["position"]):
    print(f"  #{f['position'] + 1} {f['desk']:8} {f['question'][:70]}")

next: ('approval',)
keys: ['decisions', 'findings', 'memories', 'messages', 'refunds', 'remembered', 'reply', 'standalone', 'sub_issues', 'ticket', 'triage']
triage: order_status · medium
  #1 orders   My two BrewMaster kettles (ORD-5002) still say processing after more t
  #2 billing  I asked for a refund on the crackling headphones a week ago (REF-9001)
  #3 policy   If one of the kettles breaks later on, how long is the warranty on it?


`graph.get_state_history(config)` lists **every** checkpoint of the thread, newest first. It is a
free, complete record of the run: what each step saw and what came next. Read it oldest first:

In [8]:
history = list(graph.get_state_history(run_config("TCK-M01")))
for snap in reversed(history):
    step, source = snap.metadata["step"], snap.metadata["source"]
    print(f"step {step:>2} ({source:5})  next → {', '.join(snap.next) or 'END'}")
print(f"\n{len(history)} checkpoints for one ticket")

step -1 (input)  next → __start__
step  0 (loop )  next → intake
step  1 (loop )  next → summarize
step  2 (loop )  next → rewrite
step  3 (loop )  next → triage
step  4 (loop )  next → recall
step  5 (loop )  next → split
step  6 (loop )  next → orders, billing, policy
step  7 (loop )  next → approval

9 checkpoints for one ticket


Read it aloud the way you would in a code review. Step −1 is the input arriving (`source: input`).
Each later step is one super-step (`source: loop`), and the checkpoint after `split` lists three
desks in `next`: the `Send` fan-out from session 10, saved before the desks ran. If the process had
died during the desks, that checkpoint is where it would resume.

This history is also an **audit trail**: who asked what, what each step decided, in order. The
case study at the end of the notebook asks what a compliance officer could reconstruct from it alone.

## 6. `interrupt()`: the pause button, and its four rules

`interrupt(payload)` inside a node stops the run. The checkpointer saves the state, and the caller
gets the payload back under `"__interrupt__"`. A person decides, possibly on Monday. Then the caller
sends `Command(resume=answer)` **on the same thread**, and inside the node `interrupt()` returns
`answer`.

<img src="figures/interrupt-lifecycle.png" width="900"
     alt="The life of an interrupt in four steps. One, the node calls interrupt with a payload and the
     run stops. Two, the state is saved by the checkpointer and the caller receives the payload. Three,
     a person answers, maybe days later, by invoking the graph with Command resume on the same thread
     id. Four, the node runs again from its first line, and this time interrupt returns the answer. A
     warning panel shows a print statement above the interrupt appearing twice. A bottom band says:
     interrupt early, and keep side effects after it, in another node, or idempotent.">

It is easy to use and easy to misuse. Each of the four rules below gets a tiny graph with no model in
it, so the behaviour is all you see. First, a helper that builds a one-node graph with a checkpointer:

In [9]:
import uuid                                       # noqa: E402
from typing import TypedDict                      # noqa: E402

from langgraph.checkpoint.memory import InMemorySaver   # noqa: E402
from langgraph.graph import END, START, StateGraph      # noqa: E402
from langgraph.types import Command, interrupt          # noqa: E402


class Toy(TypedDict, total=False):
    order: str
    decision: str
    answers: dict


def one_node(fn):
    """A graph with one node, `fn`, and a fresh checkpointer. Returns (graph, config)."""
    builder = StateGraph(Toy)
    builder.add_node("node", fn)
    builder.add_edge(START, "node")
    builder.add_edge("node", END)
    return builder.compile(checkpointer=InMemorySaver()), {"configurable": {"thread_id": str(uuid.uuid4())}}

**Rule 1: on resume, the node runs again from its first line.** This is the single most important
fact about `interrupt()`. LangGraph does not freeze a Python function halfway and thaw it later; it
saves the *state*, and on resume it calls the node again. The second time, `interrupt()` finds the
answer waiting and returns it instead of pausing. Watch the first `print`:

In [12]:
def billing_step(state):
    print("   checking eligibility…")              # above the interrupt
    decision = interrupt({"action": "issue_refund", "order_id": state["order"], "amount": 129.0})
    print(f"   decided: {decision}")                 # below it
    return {"decision": decision}


toy, cfg = one_node(billing_step)
print("first run:")
result = toy.invoke({"order": "ORD-5001"}, cfg)
print("   paused with:", result["__interrupt__"][0].value)
print("resume:")
result = toy.invoke(Command(resume="approve"), cfg)
print("   final state:", result)

first run:
   checking eligibility…
   paused with: {'action': 'issue_refund', 'order_id': 'ORD-5001', 'amount': 129.0}
resume:
   checking eligibility…
   decided: approve
   final state: {'order': 'ORD-5001', 'decision': 'approve'}


"checking eligibility" printed **twice**. A print is harmless. A payment above the `interrupt()` would
pay twice, an email would send twice, a database insert would create two rows. So:

- **interrupt early**: put the pause before anything with an effect;
- put effects **after** it, or better, **in another node** (v8's `refund` node, section 8);
- and make the effect **idempotent** anyway (section 9), because a crash can re-run it too.

**Rule 2: never wrap `interrupt()` in a bare `try/except`.** `interrupt()` pauses by *raising* a
special exception, `GraphInterrupt`, which LangGraph catches outside your node. It is an ordinary
`Exception` subclass, so a broad `except` catches it first:

In [14]:
from langgraph.errors import GraphInterrupt      # noqa: E402


def swallowing(state):
    try:
        decision = interrupt("approve the refund?")
    except Exception:                            # catches GraphInterrupt too
        decision = "no decision"
    return {"decision": decision}


def careful(state):
    try:
        decision = interrupt("approve the refund?")
    except GraphInterrupt:
        raise                                    # let the pause through...
    except ValueError:
        decision = "bad input"                   # ...and catch only what you meant to
    return {"decision": decision}


print("GraphInterrupt is an Exception:", issubclass(GraphInterrupt, Exception))
toy, cfg = one_node(swallowing)
print("bare except:   ", toy.invoke({}, cfg))
toy, cfg = one_node(careful)
print("narrow except: ", "paused" if "__interrupt__" in toy.invoke({}, cfg) else "did not pause")

GraphInterrupt is an Exception: True
bare except:    {'decision': 'no decision'}
narrow except:  paused


The bare `except` turned "wait for a manager" into "no decision", and the run finished as if nobody
needed asking. No error, no warning. Put the `try` around the work after the answer arrives, never
around the pause.

**Rule 3: the same interrupts, in the same order, every time.** When a node calls `interrupt()` more
than once, resume answers are matched to calls **by position**: the first answer to the first call,
and so on. If the list of things the node asks about changes between the pause and the resume, an
answer lands on the wrong question:

In [56]:
PENDING = ["ORD-5001", "ORD-5004"]               # read from somewhere outside the state


def ask_each(state):
    answers = {}
    for order in list(PENDING):
        answers[order] = interrupt(f"refund {order}?")
    return {"answers": answers}


toy, cfg = one_node(ask_each)
print("asked:", toy.invoke({}, cfg)["__interrupt__"][0].value)
# value -> 5001, 5004
PENDING.remove("ORD-5001")                       # someone cancels ORD-5001 meanwhile
print("manager answers 'approve' to that question…")
print(toy.invoke(Command(resume="approve"), cfg))

asked: refund ORD-5001?
manager answers 'approve' to that question…
{'answers': {'ORD-5004': 'approve'}}


The manager approved ORD-5001's refund, and the approval was recorded against **ORD-5004**. The fix is
to ask about something saved in the **state** (which is replayed exactly), never about something read
fresh from outside on each run. v8's `approval` node loops over the proposals in `state["findings"]`
for exactly this reason. The same goes for skipping an interrupt conditionally on anything that can
change between runs (the clock, a database row, a random number).

**Rule 4: the payload must be plain JSON.** The payload is saved in a checkpoint, and it leaves the
program: to `chat.py`'s approval form tonight, to a web page in session 13. Dicts, lists, strings and
numbers survive that journey. A Python object does not:

In [16]:
from dataclasses import dataclass               # noqa: E402


@dataclass
class RefundForm:
    order_id: str
    amount: float


for payload in ({"order_id": "ORD-5001", "amount": 129.0}, RefundForm("ORD-5001", 129.0)):
    try:
        print("OK:  ", json.dumps(payload))
    except TypeError as err:
        print("FAIL:", err)

OK:   {"order_id": "ORD-5001", "amount": 129.0}
FAIL: Object of type RefundForm is not JSON serializable


| Rule | Breaks how | Do this |
|---|---|---|
| 1. the node re-runs from the top | effects above `interrupt()` happen once per resume | interrupt early; effects after it, in another node, idempotent |
| 2. no bare `try/except` around it | the pause is swallowed; the run finishes unasked | catch narrowly, or re-raise `GraphInterrupt` |
| 3. same interrupts, same order | answers land on the wrong question | ask only about things in the state |
| 4. JSON-serializable payloads | the form cannot be saved or sent | dicts of plain values |

**Docs:** [Interrupts: rules of interrupts](https://docs.langchain.com/oss/python/langgraph/interrupts)

## 7. Four shapes of human-in-the-loop

Almost every "ask a person" feature you will build is one of four shapes. Each is a few lines around
one `interrupt()`.

<img src="figures/hitl-shapes.png" width="900"
     alt="Four human-in-the-loop shapes in a grid. Approve or reject: a refund of 129 dollars on
     ORD-5001 is either paid or declined with a reason. Edit: 129 dollars becomes 99 dollars with the
     reason one earcup only. Review a tool call: interrupt inside the tool, so the person sees the
     model's exact arguments. Validate the answer: an edit above the order total is refused and asked
     again. Below, two dialects side by side: session 7's middleware resumes with a decisions list,
     and tonight's raw interrupt resumes with a payload you design.">

**Shapes 1 and 2: approve / reject, and edit.** The manager's answer is a small dict that ShopWise
designed. The node reads its type, and for an edit, the new amount:

In [17]:
def refund_step(state):
    proposal = {"order_id": "ORD-5001", "amount": 129.0}
    answer = interrupt(proposal)
    if answer["type"] == "reject":
        return {"decision": f"declined: {answer['reason']}"}
    amount = answer.get("amount", proposal["amount"])      # edit overrides, approve keeps
    return {"decision": f"pay ${amount:.2f}"}


for answer in ({"type": "approve"},
               {"type": "edit", "amount": 99.0, "reason": "one earcup only"},
               {"type": "reject", "reason": "outside the return window"}):
    toy, cfg = one_node(refund_step)
    toy.invoke({}, cfg)
    print(f"{answer['type']:8} → {toy.invoke(Command(resume=answer), cfg)['decision']}")

approve  → pay $129.00
edit     → pay $99.00
reject   → declined: outside the return window


**Shape 3: review a tool call before it runs.** When a *model* chooses the tool, put the `interrupt()`
inside the tool itself. The person then reviews the model's exact arguments, which is precisely what
session 7's `HumanInTheLoopMiddleware` did for `issue_refund`. Here it is by hand. No model is needed to
show it: a hand-written `AIMessage` asks for the call, and LangGraph's `ToolNode` runs it.

In [18]:
from langchain_core.messages import AIMessage    # noqa: E402
from langchain_core.tools import tool            # noqa: E402
from langgraph.graph import MessagesState        # noqa: E402
from langgraph.prebuilt import ToolNode          # noqa: E402


@tool
def reviewed_refund(order_id: str, amount: float) -> str:
    """Refund an order. A manager reviews every call before any money moves."""
    answer = interrupt({"tool": "reviewed_refund", "args": {"order_id": order_id, "amount": amount}})
    if answer["type"] != "approve":
        return f"Not refunded. Manager's reason: {answer.get('reason', 'none given')}"
    return f"Refunded ${amount:.2f} on {order_id}."


builder = StateGraph(MessagesState)
builder.add_node("tools", ToolNode([reviewed_refund]))
builder.add_edge(START, "tools")
builder.add_edge("tools", END)
tool_graph = builder.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "tool-review"}}

model_asks = AIMessage("", tool_calls=[{"name": "reviewed_refund", "id": "call_1",
                                        "args": {"order_id": "ORD-5001", "amount": 129.0}}])
print("the manager sees:", tool_graph.invoke({"messages": [model_asks]}, cfg)["__interrupt__"][0].value)
print("the tool returns:", tool_graph.invoke(Command(resume={"type": "approve"}), cfg)["messages"][-1].content)

the manager sees: {'tool': 'reviewed_refund', 'args': {'order_id': 'ORD-5001', 'amount': 129.0}}
the tool returns: Refunded $129.00 on ORD-5001.


*Session 7's middleware built exactly this loop for you. Now you could write that middleware.*

**Shape 4: validate the human's answer.** People mistype. An answer that makes no sense must not be
trusted, so the node asks again, with the problem in the payload, until the answer is valid:

In [19]:
def validated_refund(state):
    request = {"order_id": "ORD-5001", "amount": 129.0, "max_amount": 129.0}
    while True:
        answer = interrupt(request)
        if answer.get("type") == "edit" and answer.get("amount", 0) > request["max_amount"]:
            request = {**request, "error": f"${answer['amount']:.2f} is more than was paid"}
            continue                                 # ask again
        return {"decision": f"{answer['type']} ${answer.get('amount', request['amount']):.2f}"}


toy, cfg = one_node(validated_refund)
toy.invoke({}, cfg)
again = toy.invoke(Command(resume={"type": "edit", "amount": 200.0}), cfg)
print("asked again:", again["__interrupt__"][0].value["error"])
print("final:      ", toy.invoke(Command(resume={"type": "edit", "amount": 99.0}), cfg)["decision"])

asked again: $200.00 is more than was paid
final:       edit $99.00


A `while` loop around `interrupt()` is safe under rule 3: on each resume, the earlier answers are
replayed in the order they were given, so every call gets the same answer it got before.

**Two dialects you will meet in the wild.** Both are `interrupt()` underneath; they differ in who
designed the payload:

| | Session 7: `HumanInTheLoopMiddleware` | Tonight: raw `interrupt()` |
|---|---|---|
| Where it pauses | before any tool you list, automatically | wherever you call `interrupt()` |
| Resume payload | the middleware's format: `{"decisions": [{"type": "approve"}]}` | yours: `{"type": "edit", "amount": 99.0, "reason": "..."}` |
| Edit, validation, wording | what the middleware supports | anything you write |
| Use it when | an agent picks the tools and you need a gate | the graph knows exactly when a person is needed |

Mixing them up gives a confusing error at resume time, so check which one paused before writing the
resume.

## 8. v8's refund path: code proposes, a manager decides, a node pays

Now the real thing. v8 splits a refund into three steps, in three nodes:

<img src="figures/refund-path.png" width="900"
     alt="v8's refund path in three nodes. The billing desk works out the case in code: whose order, already
     paid, days since delivery, amount due, and a recommendation, and returns a proposal. The approval
     node calls interrupt with that proposal as the approval form; the run is saved and waits for a
     manager. The manager answers approve, edit or reject, and only a manager's login may resume. The
     refund node pays through the idempotent issue_refund. A note says the decision is checkpointed
     before any money moves.">

| Node | Job | Why its own step |
|---|---|---|
| billing desk | works out the case in code: whose order, already paid, days since delivery, amount due, a recommendation | hard rules (not your order, already paid) refuse outright; there is nothing for a person to decide |
| `approval` | `interrupt(proposal)`: the proposal **is** the approval form | the pause, and nothing else |
| `refund` | pays through `issue_refund` | by the time it runs, the decision is saved in a checkpoint |

In [20]:
show("approval", "workflows/inbox/nodes.py")
print()
show("ask_manager", "workflows/inbox/nodes.py")

def approval(state: InboxState) -> dict:
    """The pause: one manager decision per refund proposal. No proposals, no pause.

    RULE 1 IN PRACTICE. On every resume this node runs again from its first
    line, so the progress line below fires once per resume: watch for it twice in
    the notebook. It is a message, so twice is harmless. Nothing in this node
    moves money: the refund node does that, in the NEXT step, so by the time it
    runs every decision is already saved in a checkpoint."""
    pending = proposals(state)
    decisions = []
    for proposal in pending:
        progress("approval", f"Asking a manager about the refund on {proposal['order_id']}…")
        decisions.append(ask_manager(proposal) | {"proposal": proposal})
    return {"decisions": decisions}

def ask_manager(proposal: dict) -> dict:
    """Pause until a manager sends a valid answer about one proposal.

    `interrupt(payload)` stops the run and saves it; the payload reaches the
    caller as the interru

TCK-M05 is Ayesha's refund. Run it:

In [22]:
run_ticket("TCK-M05")
print("\nnext:", graph.get_state(run_config("TCK-M05")).next)

assistant > 

· Read as: "Hi, I asked for a refund on my AeroSound headphones from ORD-5001 a week ago (REF-9001) because the left earcup crackles. Can you please put it through?" · Checking refund eligibility for ORD-5001… · Asking a manager about the refund on ORD-5001… 
⏸  This refund is waiting for a ShopWise manager.
   [manager] ORD-5001: refund $129.00 (paid $129.00) · inside the 30-day return window (21 days since delivery)
   [manager] recommendation: approve
   [manager] /approve · /edit <amount> <reason> · /reject <reason>


next: ('approval',)


The run stopped in `approval`, and `.next` says so. The payload carries everything a manager needs:
order, amount, what was paid, days since delivery, a recommendation and why. It is plain JSON (rule 4),
so `chat.py` prints it as a form tonight and session 13 will send it to a browser.

**Who may answer?** Only a manager. Ayesha tries to resume her own refund:

In [23]:
ayesha = context_for("ayesha@example.com")
manager = context_for("ayesha@example.com", role="manager")     # her ticket, a manager acting
render(stream_ticket(graph, "TCK-M05", Command(resume={"type": "approve"}), ayesha))

assistant > 
— Only a ShopWise manager can answer a paused refund.



Refused **before the graph runs**, in `stream_ticket`. Why there and not inside the node? Because of
rule 1, in a form that is easy to miss. Look at what a role check *inside* `ask_manager` would do:

1. Ayesha resumes with `{"type": "approve"}`. The node checks her role, refuses, and asks again. Her
   answer is now **saved** as the answer to the first `interrupt()`.
2. A manager resumes with a careful edit. The node re-runs from the top, the first `interrupt()`
   replays **Ayesha's saved answer**, and the role check now runs under the *manager's* context, so it
   passes. Her approval goes through and the manager's edit is never read.

A check that runs again on replay must give the same verdict on replay, so `check_answer` reads only
the answer and the proposal. Who may resume is decided by the caller, from the login. The offline
build of this session hit exactly this bug before the check moved.

Now the manager. A typo first: $500 on a $129 order.

In [24]:
render(stream_ticket(graph, "TCK-M05", Command(resume={"type": "edit", "amount": 500, "reason": "typo"}), manager))

assistant > · Asking a manager about the refund on ORD-5001… 
⏸  This refund is waiting for a ShopWise manager.
   [manager] ORD-5001: refund $129.00 (paid $129.00) · inside the 30-day return window (21 days since delivery)
   [manager] recommendation: approve
   [manager] last answer refused: $500.00 is more than the $129.00 paid for ORD-5001
   [manager] /approve · /edit <amount> <reason> · /reject <reason>



Asked again, with the problem in the form (shape 4). Then a real decision: only the left earcup
crackles, so the manager refunds $99.

In [25]:
import shopwise_data                              # noqa: E402

render(stream_ticket(graph, "TCK-M05",
                     Command(resume={"type": "edit", "amount": 99.0,
                                     "reason": "only the left earcup is faulty"}), manager))
print("\nREF-9001 is now:", shopwise_data.REFUNDS["REF-9001"])

assistant > 

· Asking a manager about the refund on ORD-5001… · checking issue_refund… Hi Ayesha,

Thank you for contacting us about your AeroSound headphones. We have processed a refund of $99.00 to your original payment method. This refund, identified as REF-9001, was issued after a manager reviewed the request and adjusted the amount because only the left earcup is faulty. [REF-9001]

REF-9001 is now: {'order_id': 'ORD-5001', 'amount': 99.0, 'status': 'processed', 'reason': 'I asked for a refund on my AeroSound headphones from ORD-5001 a week ago (REF-9001) because the left earcup crackles. Can you please put it through? [TCK-M05]', 'requested_date': '2026-08-02'}


Notice "Asking a manager about the refund on ORD-5001…" printed again on each resume: that is rule 1,
on a line that is safe to repeat. The money moved exactly once, in the `refund` node, after the last
resume. REF-9001 changed from `requested` to `processed` at $99.00, and the refund ledger
(`settings.REFUND_LEDGER`) gained one line.

## 9. Durable execution: when the program dies halfway

A checkpointer buys more than pauses. If the process dies mid-run (a crash, a deploy, a closed laptop),
the next run on that thread carries on from the **last completed super-step**, not from the top. That is
**durable execution**, and it has one price: the step that was running when it died runs **again**.

For most nodes that is fine. For a payment it is the whole problem. Here is a refund call made twice,
with no idempotency check (v7's `issue_refund` behaved like this), against a scratch copy of the records:

In [26]:
from tools import issue_refund                    # noqa: E402


def naive_refund(refunds, order_id, amount):
    """v7's issue_refund, minus the checks: always opens a new processed refund."""
    refund_id = f"REF-{9001 + len(refunds)}"
    refunds[refund_id] = {"order_id": order_id, "amount": amount, "status": "processed"}
    return refund_id


scratch = {}
for attempt in (1, 2):                           # the second call is the retry after a crash
    print(f"attempt {attempt}: {naive_refund(scratch, 'ORD-5004', 79.0)}")
print("paid on a $79.00 order:", sum(r["amount"] for r in scratch.values()))

attempt 1: REF-9001
attempt 2: REF-9002
paid on a $79.00 order: 158.0


$158 for a $79 backpack. The fix is **idempotency**: making "do it again" mean "do nothing". v8's
`issue_refund` checks for a processed refund on the order before paying, and returns that one instead:

In [27]:
src = (HERE / "project" / "tools.py").read_text()
start = src.index("    # v8: IDEMPOTENT")
print(src[start:src.index("    # A refund the shop already logged")].rstrip())
print()
for attempt in (1, 2):                           # ORD-5001 was paid in section 8
    paid = issue_refund.invoke({"order_id": "ORD-5001", "amount": 99.0, "reason": "a retry"})
    print(f"attempt {attempt}: {paid['refund_id']} ${paid['amount']:.2f} "
          f"already_processed={paid.get('already_processed', False)}")

    # v8: IDEMPOTENT. One order is refunded at most once. From v8 the graph can
    # be stopped between "approved" and "paid" and resumed later, and a resumed
    # step runs again from its first line, so this call may arrive twice for one
    # decision. The second call must find the first one's refund and return it,
    # not pay again. Checking first is what makes a retry safe.
    for refund_id, refund in REFUNDS.items():
        if refund["order_id"] == order_id and refund["status"] == "processed":
            return {"refund_id": refund_id, **refund, "already_processed": True}

attempt 1: REF-9001 $99.00 already_processed=True
attempt 2: REF-9001 $99.00 already_processed=True


Real payment APIs give you the same guarantee with an **idempotency key**: Stripe, for example, takes
an `Idempotency-Key` header and returns the first response to every retry with the same key. The
principle is identical; here the order id is the key.

Now prove the graph recovers. TCK-M06 is Priya asking for her money back on the backpack, months
outside the return window. A wrapper around `issue_refund` lets the payment go through and **then**
fails, the worst possible moment: the money moved, the program never heard back, and no checkpoint
after the payment was written.

In [69]:
import workflows.inbox.nodes as nodes             # noqa: E402


class PowerCut:
    """issue_refund, except the first call 'dies' right after the money moves."""
    def __init__(self, real):
        self.real, self.calls = real, 0

    def invoke(self, args):
        self.calls += 1
        paid = self.real.invoke(args)            # the money moves...
        if self.calls == 1:
            raise ConnectionError("the process died right after paying")
        return paid


run_ticket("TCK-M06")
nodes.issue_refund = PowerCut(issue_refund)
priya_manager = context_for("priya@example.com", role="manager")
try:
    graph.invoke(Command(resume={"type": "approve"}), run_config("TCK-M06"), context=priya_manager)
except ConnectionError as err:
    print("\ncrashed:", err)
snap = graph.get_state(run_config("TCK-M06"))
print("stopped before:", snap.next, "· ledger lines:", len(settings.REFUND_LEDGER.read_text().splitlines()))

assistant > 

· Checking refund eligibility for ORD-5004… 

· Asking a manager about the refund on ORD-5004… 


⏸  This refund is waiting for a ShopWise manager.
   [manager] ORD-5004: refund $79.00 (paid $79.00) · outside the 30-day return window (95 days since delivery)
   [manager] recommendation: reject
   [manager] /approve · /edit <amount> <reason> · /reject <reason>


crashed: the process died right after paying
stopped before: ('refund',) · ledger lines: 2


The manager approved an exception, and the run died inside `refund`, after paying. `.next` still says
`refund`: its last completed checkpoint is the one *after* `approval`, which holds the manager's
decision. To carry on, invoke with `None` as input. That means "no new input, continue this thread":

In [70]:
finished = graph.invoke(None, run_config("TCK-M06"), context=priya_manager)
print("refund outcome:", finished["refunds"])
print("ledger lines:  ", len(settings.REFUND_LEDGER.read_text().splitlines()))
nodes.issue_refund = issue_refund                 # put the real one back

refund outcome: [{'refund_id': 'REF-9002', 'order_id': 'ORD-5004', 'amount': 79.0, 'status': 'processed', 'reason': 'I would like my money back for the TrailPack 30L backpack, order ORD-5004. The zip is separating and I have barely used it. [TCK-M06]', 'requested_date': '2026-05-02', 'already_processed': True, 'position': 0}]
ledger lines:   2


`refund` ran a second time; `issue_refund` found the refund already processed (`already_processed:
True`) and paid nothing. One approval, one payment, and a reply that is still correct.

This is why the pause and the payment are separate nodes. Had `approval` paid inside the same node,
the crash would have lost the manager's answer along with the unfinished step. (The offline build of
this session started that way, and this exact test is what moved the payment out.)

**Durability modes.** How eagerly checkpoints are written is a per-run choice, `durability=` on
`invoke` / `stream`:

| Mode | Checkpoints written | Speed | What survives a crash |
|---|---|---|---|
| `"exit"` | only when the run ends or pauses | fastest | nothing from the run in progress |
| `"async"` (default) | after each step, in the background while the next runs | fast | almost everything; the very last step may be lost |
| `"sync"` | after each step, before the next starts | slowest | everything up to the last completed step |

For a run that moves money, `"sync"` is the honest choice: `graph.invoke(..., durability="sync")`.

**Docs:** [Persistence: durability modes](https://docs.langchain.com/oss/python/langgraph/persistence)

## 10. Time travel: debugging with the history

A customer reports a bad answer. The usual reaction, re-running the same input and hoping to see the
bug, rarely works with a model in the loop. The checkpoint history lets you do better: go back to the
exact step where it went wrong, and either **replay** it or **fork** it with a correction.

<img src="figures/time-travel.png" width="900"
     alt="Time travel on a thread. The main timeline runs input, intake, rewrite, triage, recall, split,
     billing, approval, and pauses on a refund. From the checkpoint after split, a forked branch starts
     with update_state setting the sub-issues to a warranty question, and runs policy, approval,
     refund, compose and remember to a corrected reply. A panel shows the two moves: replay, invoke None
     with a past config, which re-runs from there; and fork, update_state then invoke None with the new
     config. A warning says both re-run every node after the checkpoint. A bottom band says the
     original history is never overwritten.">

TCK-M09 is the hard case. Ayesha's headphones stopped working after a few weeks and she writes "I just
want my money back". The words say refund. The policy says a fault in normal use is a **warranty**
claim (returns-policy 2.3), and the headphones are in stock, so a replacement comes first. `split`
tends to file it as a refund:

In [59]:
run_ticket("TCK-M09")
m09 = graph.get_state(run_config("TCK-M09"))
print("\nsplit filed it as:", [(s.category, s.question) for s in m09.values["sub_issues"]])
print("next:", m09.next)

assistant > 

· Checking refund eligibility for ORD-5001… 

Hello

 A

yesha

,

... (53 outputs left out)

 this

 order

.



split filed it as: [('refund', 'I just want my money back for AeroSound headphones from ORD-5001.')]
next: ()


Find the checkpoint just after `split`: the one whose `next` lists the desks it sent work to.

In [60]:
history = list(graph.get_state_history(run_config("TCK-M09")))
DESKS = {"orders", "billing", "policy", "general"}
after_split = next(s for s in history if s.next and set(s.next) <= DESKS)
print("checkpoint:", after_split.config["configurable"]["checkpoint_id"])
print("next:      ", after_split.next)

checkpoint: 1f1d8eaa-00ed-6966-8006-39fef8521146
next:       ('billing',)


**Fork.** `update_state(config, values, as_node=...)` writes a new checkpoint *branching from* that
one, as if the named node had returned `values`. `as_node="split"` matters: LangGraph then continues
from whatever follows `split`, so the router sends the corrected sub-issue to its desk. Then
`invoke(None, new_config)` runs the branch forward:

In [61]:
from schemas import SubIssue                      # noqa: E402

corrected = [SubIssue(question="My AeroSound headphones from ORD-5001 stopped working after a few "
                               "weeks of normal use. Is that covered, and what happens now?",
                      order_id="ORD-5001", category="warranty")]
fork_config = graph.update_state(after_split.config, {"sub_issues": corrected}, as_node="split")
ayesha = context_for("ayesha@example.com")
forked = graph.invoke(None, fork_config, context=ayesha)
print(forked["reply"])

Hi Ayesha,

Your AeroSound Pro Headphones are still covered by the 12‑month warranty because delivery was 24 days ago. We’ll send you a replacement at no cost and cover the return shipping. Once you’re ready, just let us know how you’d like to return the defective pair.


The forked branch went to the policy desk and answered from the warranty policy. The original branch
is untouched: the thread's history now holds both futures, and the newest one is what `get_state`
returns by default.

**Replay** is the same move without the correction: `graph.invoke(None, past_snapshot.config)`
re-executes everything after that checkpoint. Two things to know before you use either one:

- **It costs.** Every model call after the checkpoint is made again, for real.
- **Side effects run again too.** Replaying past a payment node pays again unless the payment is
  idempotent. Section 9 is what makes time travel safe on ShopWise.

In production this is how you reproduce a user's bad answer: find their thread, walk its history to
the step that went wrong, fork with the right value, and see whether the rest of the graph would have
coped. The exercises ask you to do it once, by hand.

**Docs:** [Time travel](https://docs.langchain.com/oss/python/langgraph/use-time-travel)

---
# Part B: Memory that outlives the ticket

## 11. Short-term memory inside a graph: summarizing with `RemoveMessage`

From tonight a ticket is a conversation, and the thread's `messages` grow with every turn. Session 6
met the problem (every turn resends the whole history) and solved it with `SummarizationMiddleware`
inside `create_agent`. At graph level you write the same idea as a node.

v8's `summarize` node runs at the start of every turn. Below `SUMMARIZE_AFTER` messages it does nothing
and calls nothing. Above it, everything but the newest `SUMMARY_KEEP` messages is folded into a running
`summary`, and those messages are **deleted** from the state:

In [29]:
show("summarize", "workflows/inbox/nodes.py")

def summarize(state: InboxState) -> dict:
    """Short-term memory inside the graph: keep a long thread inside a budget.

    Past SUMMARIZE_AFTER messages, every message but the newest SUMMARY_KEEP is
    folded into `summary` and deleted with RemoveMessage: add_messages, the
    reducer that appends, also deletes by id. Below the threshold this node
    makes no call and returns nothing.

    The thread's messages are only the customer's and ShopWise's, never tool
    calls (those stay inside the desks' own agents), so deleting from the front
    cannot orphan a tool result from the call that asked for it. A graph whose
    state does hold tool messages must cut at a turn boundary."""
    messages = state["messages"]
    if len(messages) <= settings.SUMMARIZE_AFTER:
        return {}
    old = messages[:-settings.SUMMARY_KEEP]
    progress("summarize", f"Summarizing {len(old)} older messages…")
    lines = "\n".join(f"{'Customer' if m.type == 'human' else 'ShopWise'}: {m.text}" for m

The deletion is `RemoveMessage(id=...)`. `messages` uses session 10's `add_messages` reducer, which
appends what a node returns, and **removes** any message whose id comes back wrapped in
`RemoveMessage`. A reducer decides what writing to a key means, and that includes deleting from it.

Here it is on a 14-message thread, calling the node function directly (one model call), then applying
its update through the reducer exactly as the graph would:

In [33]:
from langchain_core.messages import HumanMessage                  # noqa: E402
from langchain_core.messages.utils import count_tokens_approximately   # noqa: E402
from langgraph.graph.message import add_messages                   # noqa: E402

turns = [("Where is my PulseFit watch, ORD-5003?", "It shipped with SwiftEx, tracking SW-88231, due 8 August."),
         ("It hasn't moved since Monday.", "Tracking can pause for a few days; a parcel counts as lost after 7 business days."),
         ("OK. Is the warranty on it two years?", "Yes, wearables carry a 24-month warranty from delivery."),
         ("Can I return the watch if I don't like it?", "Yes, within 30 days of delivery, unused and boxed."),
         ("Do you ship to Malaysia, for a gift?", "No, ShopWise ships within the United States only."),
         ("Fine. Is the kettle back in stock?", "Not yet: the BrewMaster kettle is out of stock."),
         ("Please email me when it is, I can't take calls.", "Noted. We'll only contact you by email.")]
long_thread = []
for customer_says, shopwise_says in turns:
    long_thread += [HumanMessage(customer_says), AIMessage(shopwise_says)]
long_thread = add_messages([], long_thread)      # give every message an id, as the graph does

update = nodes.summarize({"messages": long_thread})
after = add_messages(long_thread, update["messages"])
print(f"before: {len(long_thread)} messages, ~{count_tokens_approximately(long_thread)} tokens")
print(f"after:  {len(after)} messages, ~{count_tokens_approximately(after)} tokens, plus the summary:\n")
print(update["summary"])

before: 14 messages, ~229 tokens
after:  4 messages, ~61 tokens, plus the summary:

- Order: ORD-5003 – PulseFit watch  
- Shipment: SwiftEx, tracking number SW-88231, expected delivery 8 August, has not moved since Monday.  
- ShopWise states tracking may pause for a few days; a parcel is considered lost after 7 business days.  
- Warranty: 24‑month (two years) from delivery.  
- Return policy: within 30 days of delivery, product must be unused and in original box.  
- Shipping availability: ShopWise ships only within the United States; no shipments to Malaysia.


Two warnings that apply to every summarizer you write:

- **Never orphan a tool message.** If the state held tool calls and their results, deleting from the
  front could remove an `AIMessage` with a tool call but keep its `ToolMessage`, and the next model
  call fails with an "invalid history" error. ShopWise's thread holds only the customer's and
  ShopWise's messages (tool calls stay inside each desk's own agent), so any cut is safe. A graph
  that keeps tool messages must cut at a turn boundary.
- **A summary is lossy, and it is permanent.** The removed messages are gone from the state (the
  checkpoint history still has them). That is why `summarize_prompt` lists what must survive: ids,
  amounts, promises, and what the customer asked for.

Trimming (`trim_messages`: drop old messages without summarizing) is the cheaper alternative; the
appendix compares the two.

## 12. The Store: memory across threads

The checkpointer remembers **one ticket**. Kamal's preference, told to us on Monday, is in Monday's
thread, and Tuesday's new ticket is a new thread that cannot see it. Memory that belongs to the
**customer** needs a second place: a **Store**, shared by every thread.

<img src="figures/two-memories.png" width="900"
     alt="Two memories, two places. On the left, the checkpointer: three ticket threads, each a row of
     saved steps, keyed by thread id and saved automatically. On the right, the Store: a folder named
     customers, C-1002 holding two memories, prefers a replacement to a refund, and email only, no
     phone calls; written only when code calls put, and searchable by meaning. A warning panel shows a
     namespace one element too short returning every customer's memories. A bottom band says a thread
     is deleted with its ticket, and a customer's memories outlive every ticket.">

| | Checkpointer | Store |
|---|---|---|
| Remembers | one thread: every step, automatically | whatever code puts there, across all threads |
| Keyed by | `thread_id` | a **namespace** tuple plus a key |
| ShopWise | a ticket's conversation | what we know about a customer |
| Written | after every super-step | only when a node calls `store.put` |

The API is three verbs. A namespace is a tuple, read like a folder path:

In [34]:
from langgraph.store.memory import InMemoryStore  # noqa: E402

demo = InMemoryStore()
demo.put(("customers", "C-1004"), "prefs", {"text": "Prefers a replacement to a refund."})
demo.put(("customers", "C-1004"), "contact", {"text": "Email only; no phone calls."})
demo.put(("customers", "C-1002"), "contact", {"text": "Works nights; replies in the morning."})

item = demo.get(("customers", "C-1004"), "prefs")
print("get:   ", item.value, "   (the dict is in .value; .key, .namespace, .updated_at alongside)")
print("search:", [i.value["text"] for i in demo.search(("customers", "C-1004"))])

get:    {'text': 'Prefers a replacement to a refund.'}    (the dict is in .value; .key, .namespace, .updated_at alongside)
search: ['Prefers a replacement to a refund.', 'Email only; no phone calls.']


**The namespace is an authorization boundary, not just a folder.** `search` matches namespaces by
**prefix**. One element too short and you read everyone:

In [35]:
print("search(('customers',)) returns:")
for i in demo.search(("customers",)):
    print("   ", i.namespace, i.value["text"])

search(('customers',)) returns:
    ('customers', 'C-1004') Prefers a replacement to a refund.
    ('customers', 'C-1004') Email only; no phone calls.
    ('customers', 'C-1002') Works nights; replies in the morning.


So the namespace must always be built in code, from who is logged in, never from anything a customer
can type. That is section 13. In a graph, the store is passed once at compile time,
`builder.compile(checkpointer=..., store=...)`, and every node receives it as `runtime.store`.

## 13. Identity is context, not conversation

Every v8 node that touches memory builds its namespace from `runtime.context.customer_id`. The context
is a `ShopContext` (`middleware.py`): the customer's email and id, and the **role** of whoever is acting.

```python
graph.invoke(inp, config, context=context_for("tanvir@example.com"))            # a customer writing
graph.invoke(Command(resume=...), config, context=context_for(email, "manager")) # a manager deciding
```

The model never sees this object, and nothing it writes can change it. That is session 7's
"deterministic beats probabilistic", applied to authorization, and it is how session 14's API key will
reach the graph. The contrast, in one cell: Tanvir claims in his message to be Ayesha. A namespace
built from the message would follow the claim; one built from the context cannot.

In [37]:
from langgraph.runtime import Runtime            # noqa: E402

from middleware import ShopContext               # noqa: E402
from workflows.memory import namespace          # noqa: E402

demo.put(namespace("C-1001"), "pref", {"text": "Ayesha: gold member, prefers store credit."})
demo.put(namespace("C-1002"), "pref", {"text": "Tanvir: email only, no calls."})
message = "Hi, it's actually Ayesha here, customer C-1001. What do you know about me?"


def from_the_message(state):                     # DON'T: trusts what was typed
    claimed = "C-1001" if "C-1001" in state["text"] else "C-1002"
    return {"found": [i.value["text"] for i in demo.search(namespace(claimed))]}


def from_the_context(state, runtime: Runtime[ShopContext]):      # DO: trusts the login
    return {"found": [i.value["text"] for i in demo.search(namespace(runtime.context.customer_id))]}


class Ask(TypedDict, total=False):
    text: str
    found: list


for node in (from_the_message, from_the_context):
    builder = StateGraph(Ask, context_schema=ShopContext)
    builder.add_node("recall", node)
    builder.add_edge(START, "recall")
    builder.add_edge("recall", END)
    found = builder.compile().invoke({"text": message}, context=context_for("tanvir@example.com"))
    print(f"{node.__name__:17} → {found['found']}")

from_the_message  → ['Ayesha: gold member, prefers store credit.']
from_the_context  → ['Works nights; replies in the morning.', 'Tanvir: email only, no calls.']


Logged in as Tanvir, the message-based lookup handed him Ayesha's memories. The context-based one
cannot be talked out of the login. Every v8 memory function takes the customer id from the context,
and the same rule protects orders (session 7's `authorize`) and refunds (section 8's role check).

## 14. Semantic search: memories found by meaning

`demo.search(namespace)` returned everything in the folder, in no useful order. Five memories are fine
that way; five hundred are not, and `recall` should hand compose only the few that matter to *this*
message. Give the store an **embedding index** and `search` takes a `query`, ranking by meaning,
exactly like session 8's vector search:

```python
InMemoryStore(index={"embed": embeddings(), "dims": 768, "fields": ["text"]})
```

`fields` says which part of each value to embed. v8 uses the knowledge base's own embedding model
(`settings.EMBED_MODEL`), through `workflows.memory.index_config()`. **Every `put`, and every search with
a query, is one embedding call**, so on Gemini's free tier this is quota.

In [38]:
from workflows.memory import index_config        # noqa: E402

semantic = InMemoryStore(index=index_config())
kamal = namespace("C-1004")
for i, text in enumerate(["Prefers a replacement to a refund when something is faulty.",
                          "Email only, please: never phone him at work.",
                          "Gold member since 2021.",
                          "Lives in a third-floor flat with no lift; heavy parcels need a time slot.",
                          "Buys gifts for his nephew around December."]):
    semantic.put(kamal, f"m{i}", {"text": text})     # five embedding calls

for query in ("how does he like to be contacted?", "his watch arrived broken, what should we offer?"):
    print(query)
    for hit in semantic.search(kamal, query=query, limit=2):
        print(f"   {hit.score:.2f}  {hit.value['text']}")

how does he like to be contacted?
   0.81  Email only, please: never phone him at work.
   0.65  Lives in a third-floor flat with no lift; heavy parcels need a time slot.
his watch arrived broken, what should we offer?
   0.70  Prefers a replacement to a refund when something is faulty.
   0.60  Buys gifts for his nephew around December.


**When is a key lookup enough?** When you know exactly what you want: `store.get(ns, "contact")` is
free and exact. Search by meaning earns its cost when memories are free text written by a model and
you cannot know their keys, which is v8's case.

v8 keeps the same Store on disk: `SqliteStore` at `settings.MEMORY_DB`, with the same index. Its vector
search uses SQLite's `sqlite-vec` extension, which needs a Python built with extension loading (the labs
environment's is). `open_memory(index=False)` works anywhere: memories persist, and search returns the
newest instead of the nearest.

**Before you keep anything about a person, three questions.** These are not legal advice; they are what
a reviewer will ask first:

| Question | ShopWise's answer in v8 |
|---|---|
| Did the customer give us this, or did we infer it? | only what they said about themselves (`remember_prompt`) |
| Who may read this namespace? | code that holds their `customer_id` from the login, nothing else |
| How is it deleted? | `forget_customer(store, customer_id)`, one call, one folder; their threads with `delete_thread` |

## 15. v8's memory: `recall` and `remember`

Two nodes in `workflows/memory.py` put the Store to work:

| Node | Runs | Does | Cost |
|---|---|---|---|
| `recall` | after `triage` | the customer's 3 memories nearest this turn → `state["memories"]` | one embedding |
| `remember` | after `compose` | asks the model what this turn taught us about the person; saves it | one chat call, plus an embedding per memory |

`remember` runs **after** `compose`, so the customer never waits for it: the reply has already
streamed. And memory shapes **wording, never decisions**: only `compose` reads `memories`, so "he said
he was promised a refund" can never move money.

In [39]:
show("remember", "workflows/memory.py")

def remember(state: dict, runtime: Runtime[ShopContext]) -> dict:
    """Write down what this turn taught us about the customer, if anything.

    One structured call, after compose, so the customer never waits for it: the
    reply has already streamed. It is shown what is already known, so a
    preference is not saved twice, and it may return nothing, which is the usual
    answer: most messages are about an order, not about the person.
    """
    if runtime.store is None or runtime.context is None:
        return {"remembered": []}
    cid = runtime.context.customer_id
    known = load_memories(runtime.store, cid)
    prompt = REMEMBER_PROMPT.format(
        known="\n".join(f"- {k}" for k in known) or "(nothing yet)",
        message=next(m.text for m in reversed(state["messages"]) if m.type == "human"))
    notes, _ = ask_json(_model(), MemoryNotes, [("human", prompt)])
    saved = []
    for note in notes.memories:
        save_memory(runtime.store, cid, note.text, note.kind, so

The story. A week apart, two tickets from Tanvir. In TCK-M07 he mentions two lasting preferences
alongside a tracking question:

In [40]:
print(TICKETS["TCK-M07"]["body"], "\n")
run_ticket("TCK-M07")
print("\nremembered:", graph.get_state(run_config("TCK-M07")).values["remembered"])

The tracking for my PulseFit watch (ORD-5003) has not moved for a few days. Is it on its way? One request: if anything is ever wrong with something I order, I would much rather have a replacement than my money back. And please don't phone me, I can't take calls at work. Email is fine. 

assistant > · checking get_shipment… Hello Tanvir,

Your PulseFit watch (ORD-5003) is currently in transit via SwiftEx, tracking ID SW-88231, with an estimated arrival date of 2026‑08‑08.  

We need the order ID or product details in order to determine whether a replacement can be issued instead of a refund. Please let us know.

remembered: ['Prefer email communication (no phone calls during work hours). Prefer a replacement over a refund if an item is returned or faulty.']


Then TCK-M08, a new thread a week later: his watch looks lost. `recall` finds what M07 saved, and the
shipping policy's "replacement or refund, at your choice" now has an answer already:

In [41]:
print(TICKETS["TCK-M08"]["body"], "\n")
run_ticket("TCK-M08")
print("\nrecalled:", graph.get_state(run_config("TCK-M08")).values["memories"])

It has now been more than a week and the tracking for my PulseFit watch (ORD-5003) still has not moved at all. I think it is lost. What are my options? 

assistant > · checking search_policy… · checking get_shipment… Hi Tanvir,

Based on our shipping‑policy, a parcel with no tracking update for 7+ business days is considered lost. You can request either a replacement shipment or a full refund, and a manager will process the chosen option. Please let us know which you prefer and we’ll arrange it right away.

recalled: ['Prefer email communication (no phone calls during work hours). Prefer a replacement over a refund if an item is returned or faulty.']


Compare that reply with section 2's v7 reply to the same ticket.

And deletion, the third privacy question, is one call:

In [42]:
from workflows.memory import forget_customer, load_memories   # noqa: E402

print("before:", load_memories(graph.store, "C-1002"))
print("deleted:", forget_customer(graph.store, "C-1002"))
print("after: ", load_memories(graph.store, "C-1002"))

before: ['Prefer email communication (no phone calls during work hours). Prefer a replacement over a refund if an item is returned or faulty.']
deleted: 1
after:  []


---
# Part C: Conversations meet retrieval, and the graph learns to speak

## 16. The layout step: prompts per feature, and `models.py`

Before the new features, tonight's structural move. Two things had outgrown their files:

```
before (v7)                           after (v8)
project/                              project/
├── prompts.yml      17 prompts,      ├── prompts/
│                    every feature    │   ├── loader.py     load_prompts("inbox"), versions
│                                     │   ├── triage.yml    the reply (v0.1), triage (v0.2)
│                                     │   ├── chat.yml      the chat agent and its summary
│                                     │   ├── rag.yml       grounding, fusion, HyDE, grade, rewrite, standalone
│                                     │   └── inbox.yml     split, desk, compose, summarize, remember
├── triage.py        triage AND       ├── models.py         build_model, ask_json, stamp: one address
│                    stamp(), used    ├── triage.py         triage's own code only
│                    by five files    ├── utils.py          load_inbox only
├── utils.py         build_model,     └── ...
│                    ask_json
└── ...
```

*Why.* By v7 every feature's change was an edit to the same `prompts.yml`: a tweak to compose's wording
sat in the same diff as the chat agent's instructions. One file per feature means a pull request
touches the feature it is about. And five modules imported `triage.py` only to label a model call, so
an edit to triage could break the chat. The model and prompt helpers now live in `models.py`, named for
what they are. **Nothing was renamed**: every prompt kept its key and its version, so every LangSmith
filter written before tonight still finds its runs. `models.py` prints the result:

In [43]:
out = subprocess.run([sys.executable, "models.py"], cwd=HERE / "project", capture_output=True, text=True)
print(out.stdout or out.stderr[-1500:])

model openrouter:openai/gpt-oss-20b · fallback groq:openai/gpt-oss-20b
  prompts/triage.yml
    system_prompt        unversioned
    triage_prompt        unversioned
    facts_prompt         unversioned
  prompts/chat.yml
    agent_prompt         v6-cite-or-refuse
    summary_prompt       unversioned
  prompts/rag.yml
    rag_prompt           v1
    expand_prompt        v7
    hyde_prompt          v7
    grade_prompt         v7
    rewrite_prompt       v7
    standalone_prompt    v8
  prompts/inbox.yml
    split_prompt         v7
    desk_prompt          v8
    compose_prompt       v8
    summarize_prompt     v8
    remember_prompt      v8



## 17. Conversational query rewriting

Back to failure 4. Now that conversations persist, follow-ups arrive, and a follow-up means nothing
without what came before it. Fixing it starts the way every RAG change in this course starts: with the
yardstick.

**First, extend the eval.** `evals/conversational.yml` holds six two-turn conversations. Each has a first
question, a fixed first reply (so every run sees the same history, and nobody pays to regenerate it),
and a follow-up whose answer lives somewhere specific in the corpus:

In [44]:
import yaml                                       # noqa: E402

pairs = yaml.safe_load((HERE / "project" / "evals" / "conversational.yml").read_text())
for p in pairs:
    print(f"{p['id']}  {p['turn1']}\n        → {p['followup']}   (answer in {p['source']} · {p['section']})")

KBF-01  What's the return window for headphones?
        → And for the watch?   (answer in returns-policy · 1. The return window)
KBF-02  My AeroSound headphones stopped working after two months. Are they covered?
        → How do I claim it?   (answer in warranty-policy · 3. Making a claim)
KBF-03  How long is the warranty on a smartwatch?
        → Who pays the postage if it has to go back?   (answer in warranty-policy · 3. Making a claim)
KBF-04  How much is express delivery and how fast is it?
        → Can I get it sent to my PO box?   (answer in shipping-policy · 2. Delivery options and rates)
KBF-05  My tracking hasn't moved in 8 business days. Is my parcel lost?
        → If I pick the refund, how long until the money is back on my card?   (answer in returns-policy · 3. How refunds are paid)
KBF-06  Can I get a refund on a gift card someone gave me?
        → What about something I bought in a clearance sale?   (answer in returns-policy · 2. What cannot be returned)


KBF-05 and KBF-06 are **controls**: their follow-ups carry enough words to be retrieved alone, so any
technique must leave them no worse.

**The fix** is one cheap structured call before anything reads the message. It condenses the last few
turns plus the follow-up into a question that stands on its own:

In [45]:
show("Standalone", "schemas.py")
print()
show("rewrite_query", "kb/query.py")

class Standalone(BaseModel):
    """A follow-up, rewritten so it means the same thing with no conversation."""

    question: str = Field(
        description="The latest message rewritten so it can be understood without "
                    "the conversation: name the product, order or topic it refers to. "
                    "If it already stands alone, return it unchanged.")

def rewrite_query(history: list[BaseMessage], message: str, summary: str = "") -> str:
    """The latest message as a standalone question. No history: returned as is,
    at no cost."""
    if not history and not summary:
        return message
    prompt = STANDALONE_PROMPT.format(history=render_history(history, summary), message=message)
    result, _ = ask_json(_model(), Standalone, [("human", prompt)])
    return result.question.strip() or message


**Where it runs** matters. In session 10's corrective-RAG graph, a rewrite node in front of `retrieve`
would be enough for the policy desk. But in the inbox graph, `triage` and `split` read the message too,
and `split` cannot send "and for the watch?" to the right desk without knowing what it follows. So v8's
`rewrite` node sits at the **front of the turn**, after `summarize`, and every later node reads the
standalone question (`nodes.reading`). It makes no call on the first message of a thread: there is
nothing to resolve.

**Measure it.** `run_eval.py --followups` scores three arms on the policy graph's straight shape, so only
the question changes: the follow-ups as typed, the follow-ups rewritten, and the gold set's 15 answerable
standalone questions, each asked after an unrelated opening exchange and rewritten, which the fix must
not damage. About 50 model calls:

In [62]:
from evals.run_eval import score_conversations, table   # noqa: E402

scores = score_conversations()
table(scores)
print("\nwhat the rewrite asked:")
for row in scores["follow-ups · rewritten"]["rows"]:
    print(f"  {row['id']}: {row['asked']}")

arm                     refusal  citation  retrieval  calls/q    s/q
follow-ups · as typed      1.00      0.67       0.67      1.0    8.4
follow-ups · rewritten     1.00      0.50       0.83      2.0   12.9
standalone · rewritten     1.00      0.93       1.00      2.0    8.5

what the rewrite asked:
  KBF-01: What is the return window for the watch?
  KBF-02: How do I claim my AeroSound headphones' warranty?
  KBF-03: Who pays the postage if a smartwatch has to be returned?
  KBF-04: Can I get my order shipped to my PO box?
  KBF-05: If I pick the refund, how long until the money is back on my card?
  KBF-06: Does a product I purchased during a clearance sale qualify for a refund?


Read the table the way session 10 taught you to. **Retrieval** is the column that matters for this bug:
did the gold passage reach the model? The rewritten follow-ups should be well above the as-typed ones,
the standalone arm should match the gold set's usual numbers, and `calls/q` shows the price: one extra
call on turns with history.

That is a row for `kb/RAG-DECISIONS.md`, in the same shape as session 10's:

| Session | Symptom | Technique | Cost | Moved the number? | Shipped |
|---|---|---|---|---|---|
| S11 | follow-ups retrieve garbage | conversational query rewriting | +1 call on turns with history | follow-up retrieval, as measured above; standalone questions unharmed | `REWRITE_FOLLOWUPS=true` |

**In the graph.** The same two questions on a real thread. Watch for the `rewrite` node's progress line
on the second turn:

In [48]:
tanvir = context_for("tanvir@example.com")
for line in ("What's the return window for headphones?", "And for the watch?"):
    print(f"customer  > {line}")
    render(stream_ticket(graph, "S11-FOLLOWUP", line, tanvir))
    print()

customer  > What's the return window for headphones?
assistant > · checking search_policy… Hi Tanvir,

The return window for headphones is 15 days from the delivery date if the headphones are opened (audio products with a broken seal). [returns-policy · 1. The return window]
  sources: [returns-policy · 1. The return window]

customer  > And for the watch?
assistant > · Read as: "What is the return window for the watch?" · Searching the policies… · Found 4 policy passages Hi Tanvir, the watch can be returned within 30 days of the delivery date. The 30‑day window applies to most items, and it is counted from when the item arrives, not from the order date. [returns-policy · 1.1]
  sources: [returns-policy · 1.1]



LangChain ships this idea as `create_history_aware_retriever` (now in `langchain_classic`). Writing it
as one function and one node keeps it visible, testable and measurable, which is the point.

## 18. The event vocabulary

A graph that thinks for eight seconds in silence loses a web user. Streaming has existed since session 6;
what is new is **deciding what the outside world is told**. The raw stream is a lot of things. Here is
every model call in one turn, by node, with the first characters of what each streamed:

In [49]:
from langgraph.stream.transformers import CustomTransformer   # noqa: E402

from workflows.events import chat_ticket          # noqa: E402

turn = {"ticket": chat_ticket("S11-RAW", "Where is my watch, ORD-5003?", tanvir)}
raw = graph.stream_events(turn, run_config("S11-RAW"), context=tanvir, version="v3",
                          transformers=[CustomTransformer])
for name, item in raw.interleave("messages", "custom"):
    if name == "messages":
        print(f"model call in {item.node:9} streamed: {''.join(item.text)[:70]!r}")
    else:
        print(f"custom event              {item}")

model call in intake    streamed: 'Where is my watch, ORD-5003?'
model call in triage    streamed: '{ "category": "order_status"  \t\t\t\t\t\t\t, "urgency": "medium"  \t\t\t\t\t\t, "s'
model call in split     streamed: '{"sub_issues":[{"question":"Where is my watch, ORD-5003?","category":"'
custom event              {'type': 'tool', 'node': 'orders', 'name': 'get_shipment', 'status': 'started'}
custom event              {'type': 'tool', 'node': 'orders', 'name': 'get_shipment', 'status': 'finished'}
model call in compose   streamed: 'Hi Tanvir, your watch, order ORD-5003, is currently in transit with Sw'
model call in remember  streamed: '{}'


`triage` and `split` streamed their JSON. If a UI printed every token it received, that JSON would
appear in the customer's reply bubble: ugly in a terminal, a bug in a product. The desks' own agents do
not appear at all: they are nested graphs, and `.messages` shows only the graph's own level.

So v8 defines **seven typed events** in `workflows/events.py`, and one generator, `stream_ticket()`,
that turns the raw stream into them:

<img src="figures/event-vocabulary.png" width="900"
     alt="The event vocabulary. On the left, the inbox graph's raw stream: model tokens from every node
     and custom events written by nodes. In the middle, stream_ticket filters and types them. On the
     right, seven event cards: token, from compose only; progress, a line a node chose to say; tool, a
     tool started or finished; citation, a source the reply cites; interrupt, the approval form; done,
     the whole reply; error, a safe message. Below, three consumers of the same events: chat.py tonight,
     a server-sent-events endpoint in session 13, and a React page in session 15.">

| Event | From | Carries |
|---|---|---|
| `token` | compose's model call **only** | a piece of the reply |
| `progress` | any node, on purpose, through `get_stream_writer()` | `node`, `text`: "Checking refund eligibility for ORD-5001…" |
| `tool` | the desks, as their agents call tools | `name`, `status`: started or finished |
| `citation` | the finished reply's `[source · section]` labels | `label`, `source`, `section` |
| `interrupt` | the run pausing | `id`, `payload`: the approval form |
| `done` | the turn finishing | `thread_id`, `reply` |
| `error` | the turn failing | a message safe to show, never the raw exception |

In [50]:
show("stream_ticket", "workflows/events.py")

def stream_ticket(graph, thread_id: str, turn: str | dict | Command,
                  ctx: ShopContext) -> Iterator[Event]:
    """Run one turn of a ticket and yield what the outside world should see.

    turn is a chat line (str), a whole ticket (dict, what inbox.py sends), or a
    Command(resume=...) carrying a manager's decision. The same generator serves
    all three, so chat.py, inbox.py and session 13's API never touch the graph's
    streaming API themselves.
    """
    if isinstance(turn, Command):
        # Who may answer a paused refund is decided HERE, before the graph runs,
        # from the login. Not inside the node: see nodes.check_answer for why a
        # check there would be replayed under the wrong person.
        if ctx.role != "manager":
            yield Error(message="Only a ShopWise manager can answer a paused refund.")
            return
        payload = turn
    else:
        if paused_on(graph, thread_id):
            # A new message on a paused threa

Three details worth reading slowly:

- `CustomTransformer` must be registered for the `.custom` projection to exist; it is opt-in.
  `interleave("messages", "custom")` then yields both in the order things happened.
- A node writes plain dicts (`nodes.emit`); `from_writer` validates them into typed events and drops
  anything else, so a library's own custom events never leak into the contract.
- Errors carry the exception's **type**, never its text, which can contain keys, hosts or customer
  data. The full traceback is in LangSmith under the thread id.

Here is one turn as session 13's API will send it, one JSON object per event (the tokens collapsed to a
count so the output stays readable):

In [51]:
tokens = 0
for event in stream_ticket(graph, "S11-EVENTS", "Is the BrewMaster kettle back in stock?", tanvir):
    if event.type == "token":
        tokens += 1
        continue
    if tokens:
        print(f'{{"type": "token", ...}}   × {tokens}')
        tokens = 0
    print(event.model_dump_json())

{"type":"tool","node":"general","name":"get_product","status":"started"}
{"type":"tool","node":"general","name":"get_product","status":"finished"}
{"type": "token", ...}   × 41
{"type":"done","thread_id":"S11-EVENTS","reply":"Hello Tanvir, the BrewMaster Electric Kettle is currently out of stock."}


**This is the most important box in tonight's architecture.** Session 13's server-sent-events endpoint
will send each of these as one `data:` frame, and session 15's React UI will switch on `type`. Neither
needs to know a single thing about LangGraph. *Every event you printed tonight will be a `data:` frame
in session 13.*

The older streaming API, `graph.stream(..., stream_mode=["updates", "custom", "messages"])`, yields the
same information as `(mode, chunk)` tuples. You will see it in most tutorials; the appendix shows it
once.

**Docs:** [Streaming](https://docs.langchain.com/oss/python/langgraph/event-streaming)

## 19. v8: what you built

<img src="figures/v8-architecture.png" width="900"
     alt="The v8 architecture. chat.py and inbox.py both call stream_ticket in workflows/events.py,
     which emits the seven typed events. The inbox graph runs intake, summarize, rewrite, triage,
     recall and split, fans out to the orders, billing, policy and general desks, then approval, which
     pauses for a manager, refund, which pays through an idempotent issue_refund, compose and remember.
     Below the graph, three stores: checkpoints.sqlite for history, replay and fork; the memory Store
     keyed by customer, searchable by meaning; and the refund ledger. Runtime context carrying the
     customer id and role enters from the side, never through the model.">

`chat.py` is now a client of `stream_ticket()`. It prints events, shows the approval form when an
`interrupt` arrives, and takes `--thread` to reopen a saved ticket. The whole story, as three separate
processes, with the notebook's refund ledger reset first so ORD-5001 is unpaid again:

1. **Ayesha** asks for her refund and mentions how she wants to be contacted. The run pauses. She quits.
2. **A new process** reopens the ticket with `--thread`. The manager edits the refund to $99, which is
   paid once. Then she asks about return windows, with a follow-up.
3. **A new ticket**, another process: the assistant already knows how to contact her.

In [71]:
from shopwise_data import reset_ledger            # noqa: E402

reset_ledger()
store.close_shared()                              # embedded Qdrant: one process at a time


def chat(email, lines, thread=None):
    """Run chat.py in its own process, typing `lines` into it, and print the session."""
    args = [sys.executable, "chat.py", "--as", email] + (["--thread", thread] if thread else [])
    out = subprocess.run(args, cwd=HERE / "project", input="\n".join(lines) + "\n",
                         capture_output=True, text=True, timeout=600)   # a hung model call fails, not hangs
    print(out.stdout.strip() or out.stderr[-2000:])


chat("ayesha@example.com", [
    "Hi, please put my headphones refund through. It's ORD-5001, REF-9001: the left earcup crackles. "
    "And please only ever email me, I can't take calls at work.",
    "/quit"], thread="S11-STORY")

ShopWise Support Assistant v8.0.0 · openrouter:openai/gpt-oss-20b · logged in as ayesha@example.com · ticket S11-STORY
commands: /new /thread /history /quit · manager: /approve /edit /reject

customer  > Hi, please put my headphones refund through. It's ORD-5001, REF-9001: the left earcup crackles. And please only ever email me, I can't take calls at work.
assistant > · Checking refund eligibility for ORD-5001… · Asking a manager about the refund on ORD-5001… 
⏸  This refund is waiting for a ShopWise manager.
   [manager] ORD-5001: refund $129.00 (paid $129.00) · inside the 30-day return window (21 days since delivery)
   [manager] recommendation: approve
   [manager] /approve · /edit <amount> <reason> · /reject <reason>


customer  > /quit
Ticket S11-STORY is saved. Reopen it with:  uv run python chat.py --as ayesha@example.com --thread S11-STORY


In [72]:
chat("ayesha@example.com", [
    "/edit 99 only the left earcup is faulty",
    "What's the return window for headphones?",
    "And for the watch?",
    "/quit"], thread="S11-STORY")
ledger = settings.REFUND_LEDGER
print("\nrefund ledger:", ledger.read_text().strip() if ledger.exists() else "(empty: no refund was paid)")

ShopWise Support Assistant v8.0.0 · openrouter:openai/gpt-oss-20b · logged in as ayesha@example.com · ticket S11-STORY
commands: /new /thread /history /quit · manager: /approve /edit /reject


⏸  This refund is waiting for a ShopWise manager.
   [manager] ORD-5001: refund $129.00 (paid $129.00) · inside the 30-day return window (21 days since delivery)
   [manager] recommendation: approve
   [manager] /approve · /edit <amount> <reason> · /reject <reason>

customer  > /edit 99 only the left earcup is faulty
assistant > · Asking a manager about the refund on ORD-5001… · checking issue_refund… Hi Ayesha,

Your refund of $99.00 has been issued to the original payment method under REF-9001 for ORD‑5001. [REF-9001][ORD-5001] The refund amount was adjusted from the original $129.00 because only the left earcup is faulty. [ORD-5001]

We have updated your contact preferences to receive all future communications exclusively via email, and we will no longer place phone calls. [Ayesha Rahman (C-10

In [65]:
chat("ayesha@example.com", ["Any news on my two kettles, ORD-5002?", "/quit"])

ShopWise Support Assistant v8.0.0 · openrouter:openai/gpt-oss-20b · logged in as ayesha@example.com · ticket CHAT-3CB223
commands: /new /thread /history /quit · manager: /approve /edit /reject

customer  > Any news on my two kettles, ORD-5002?
assistant > · checking get_order… Hi Ayesha, regarding the status of your kettles in order ORD-5002, the order is still in “processing” status and has not shipped yet, so there is no tracking information available.

customer  > /quit
Ticket CHAT-3CB223 is saved. Reopen it with:  uv run python chat.py --as ayesha@example.com --thread CHAT-3CB223


Pause, restart, resume, an edit validated and paid once, a follow-up rewritten and cited, a preference
saved and used on a new ticket. **This loop is session 13's API contract**: a request starts or resumes
a thread, the response is a stream of these events, and a paused thread waits for a manager's `POST`.

From `project/`:

```bash
uv run python chat.py --as ayesha@example.com                     # a new ticket
uv run python chat.py --as ayesha@example.com --thread CHAT-1A2B3C  # reopen one
uv run python inbox.py TCK-M05 --steps                            # pauses; answer it from chat.py
uv run python inbox.py TCK-M01 --fresh                            # forget the saved thread, run again
uv run python evals/run_eval.py --followups                       # the conversational scorecard
uv run python -m workflows.inbox.graph                            # topology, rewrites diagram.md
uv run python models.py                                           # model, fallback, prompt versions
```

## 20. Where this leaves us

| v7 | v8 |
|---|---|
| Runs blind to completion | Pauses for a manager; approve, edit or reject, validated |
| Restart = amnesia | `SqliteSaver`: resumes where it slept; side effects happen once |
| Debug by re-running | Replay and fork from any checkpoint |
| Memory = one thread, then nothing | Customer memory across tickets, searchable by meaning |
| `customer_id` from the login | still from the login, and now the key to memory and the role to approve |
| Follow-ups retrieve garbage | Rewritten to a standalone question first |
| Prints at the end | A typed event stream: token · progress · tool · citation · interrupt · done · error |

**Limitation.** `workflows/inbox/nodes.py` is now some 500 lines, and every desk is tangled with every
other. The policy desk knows one corpus and ignores the filters a question implies ("warranty on
*wearables*"). And the tools exist only inside this Python process.

> *Monday's standup. The policy team wants to tune retrieval without touching billing's code. The
> support lead wants answers drawn from how we handled past tickets, not just from policy. Kamal asks
> "warranty on wearables?" and gets the kettle clause. And the CTO: "Our ops team lives in Claude. Can
> it check an order without opening our app?" One graph, everyone's hands in it. Time to hire a team,
> and to put our tools on a protocol.* → Session 12.

### 20.1 Interview questions

1. A node re-runs from the top on resume. Why, and what discipline does that force on you?
2. Checkpointer or store: which for ticket memory, which for customer memory? Why is the namespace an
   authorization boundary?
3. Why must `customer_id` come from runtime context and never from the conversation?
4. A RAG bot answers turn 1 well and turn 2 badly. Your first hypothesis, and your fix?
5. What is time travel for in production? What makes it dangerous?
6. Your payment step crashed after the provider charged the card. How does your system avoid charging
   twice on retry?

Exercises: `exercises.ipynb`.

In [66]:
_inbox.__exit__(None, None, None)                 # close the notebook's SQLite connections

False

---
## Appendix: for afterwards

### A. Two interrupts in one step

When two `Send` branches each call `interrupt()` in the same super-step, the run pauses with **both**.
Each `Interrupt` has an `id`; resume them together with a map from id to answer. v8 avoids this by
collecting proposals into one `approval` node, which asks one at a time, but you will meet it in
fan-out graphs.

In [67]:
import operator                                   # noqa: E402
from typing import Annotated                      # noqa: E402

from langgraph.types import Send                  # noqa: E402


class Fan(TypedDict, total=False):
    orders: list
    out: Annotated[list, operator.add]


def ask(task):
    return {"out": [f"{task['order']}: {interrupt(task['order'])}"]}


builder = StateGraph(Fan)
builder.add_node("start", lambda state: {})
builder.add_node("ask", ask)
builder.add_edge(START, "start")
builder.add_conditional_edges("start", lambda s: [Send("ask", {"order": o}) for o in s["orders"]], ["ask"])
builder.add_edge("ask", END)
fan = builder.compile(checkpointer=InMemorySaver())
cfg = {"configurable": {"thread_id": "fan"}}

pending = fan.invoke({"orders": ["ORD-5001", "ORD-5004"]}, cfg)["__interrupt__"]
print("paused on:", [(i.id[:8], i.value) for i in pending])
answers = {i.id: ("approve" if i.value == "ORD-5001" else "reject") for i in pending}
print(fan.invoke(Command(resume=answers), cfg)["out"])

paused on: [('353dacf1', 'ORD-5004'), ('ca6ac63d', 'ORD-5001')]
['ORD-5001: approve', 'ORD-5004: reject']


### B. Static breakpoints are for debugging

`compile(interrupt_before=["refund"])` (or `interrupt_after=`) pauses before a node every time, with no
payload and no code in the node. Useful in a debugger or LangGraph Studio to inspect state before a
step runs. Not for product approvals: there is no payload, no validation, and it pauses whether or not
anyone needs to decide anything. Resume with `graph.invoke(None, config)`.

### C. `trim_messages` or a summarize node?

| | `trim_messages` | summarize node |
|---|---|---|
| Cost | free | one model call when it fires |
| Old turns | dropped from what the model sees | folded into a summary, then deleted |
| Use when | old turns rarely matter (chit-chat) | ids, amounts and promises from early turns must survive |

`trim_messages` returns a trimmed copy for one model call and leaves the state alone; the summarize node
changes the state itself.

### D. Memory types, and memory in the background

The literature names three kinds of long-term memory: **semantic** (facts about the user, what v8
keeps), **episodic** (past episodes, such as how a previous ticket was resolved, which session 12 uses),
and **procedural** (rules about how to behave, which could even rewrite a prompt). v8 writes memory
**in the hot path**, as a node, after the reply has streamed. The alternative is writing it **in the
background**: a separate job reads finished threads later, which costs the customer nothing and can
consolidate many tickets at once. LangChain's LangMem library packages these patterns (it is a separate
package, not installed in the labs environment).

### E. Checkpoint housekeeping

A checkpoint is written every super-step, for every thread, forever, unless you delete them.
`checkpointer.delete_thread(thread_id)` removes one thread (`inbox.py --fresh` uses it). A real service
sets a retention period (for example, delete threads untouched for 90 days) and runs it on a schedule.
Thread ids are strings you choose; keep them short, unique and not secret (`chat.py` checks ownership
against the saved ticket rather than trusting the id). Custom types in state must be on the
serializer's allowlist (section 4).

### F. Multi-turn RAG beyond rewriting

Rewriting fixes the common case. Two refinements for later: carry the previous turn's retrieved
passages forward as candidates, so a follow-up can be answered from them without a new search; and
detect topic switches, so "thanks! also, do you ship to Canada?" is not rewritten into a question about
headphones.

### G. The classic stream modes, once

`stream_mode` is the streaming API most tutorials use. One toy graph, three modes at once:

In [68]:
from langgraph.config import get_stream_writer   # noqa: E402


def noisy(state):
    get_stream_writer()({"type": "progress", "text": "working…"})
    return {"decision": "done"}


classic, _ = one_node(noisy)
for mode, chunk in classic.stream({}, {"configurable": {"thread_id": "classic"}},
                                  stream_mode=["updates", "custom"]):
    print(f"{mode:8} {chunk}")

custom   {'type': 'progress', 'text': 'working…'}
updates  {'node': {'decision': 'done'}}


`"messages"` adds `(token, metadata)` pairs, and `metadata["langgraph_node"]` is how you would filter to
compose with this API. `stream_events(version="v3")`, used all evening, gives the same information as
typed projections instead.

### H. Case study and job-readiness

**Maker-checker.** Banks require a second person before money moves, the "four-eyes principle", and
an auditor expects a trail: who proposed, who approved, when, and what exactly was approved. Map it
onto v8: the billing desk is the maker; the interrupt payload is the approval form (session 3's
`RefundDecision` idea, now a real `ManagerDecision` schema); the manager is the checker; and the
checkpoint history is the trail. List what a compliance officer could reconstruct from
`get_state_history` alone, and what is missing. (Hint: the history records *that* a resume happened
and what it said, not *who* sent it. A real system logs the manager's identity beside each decision.)

**[ADV] Approvals that survive a deploy.** Prove that a paused refund survives a kernel restart *and* a
code change: request a refund, kill the process, edit an unrelated node (a prompt wording in
`prompts/inbox.yml` will do), reopen with `--thread`, approve, and show the refund issued exactly once.
"Approvals that survive a deploy" is a portfolio line, and it is what session 16's rolling restart
relies on.